# Background

The following may be helpful when reading or debugginfg this notebook:


### Standard Amino Acids
FASTA represents each amino acid with a single character as follows:

| 1-Letter Code | 3-Letter Code | Amino Acid Name |
|---|---|---|
| A | Ala | Alanine |
| C | Cys | Cysteine |
| D | Asp | Aspartic acid |
| E | Glu | Glutamic acid |
| F | Phe | Phenylalanine |
| G | Gly | Glycine |
| H | His | Histidine |
| I | Ile | Isoleucine |
| K | Lys | Lysine |
| L | Leu | Leucine |
| M | Met | Methionine |
| N | Asn | Asparagine |
| P | Pro | Proline |
| Q | Gln | Glutamine |
| R | Arg | Arginine |
| S | Ser | Serine |
| T | Thr | Threonine |
| V | Val | Valine |
| W | Trp | Tryptophan |
| Y | Tyr | Tyrosine |

### Ambiguous & Special Characters
FASTA also uses the following abbreviations for ambiguous amino acid identification and special characters:

| 1-Letter Code | Description / Meaning |
|---|---|
| B | Aspartic acid (D) or Asparagine (N) |
| J | Leucine (L) or Isoleucine (I) |
| X | Unknown or any amino acid |
| Z | Glutamic acid (E) or Glutamine (Q) |
| * | Translation stop codon |
| - | Gap of missing or unsequenced amino acid |

# Environment Setup

In this section we're setting up the environment for the rest of the notebook.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import warnings
import zipfile
import random
import torch
import esm
import re

# File read / write
from types import resolve_bases
import pickle

# Machine Learning / Modeling
from transformers import AutoTokenizer, EsmModel
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (mean_absolute_error, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, 
                             average_precision_score)

# Gradient boosting libraries
import lightgbm as lgb
import xgboost as xgb
from xgboost import XGBClassifier

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

# Statistical / other utilities
from scipy.stats import spearmanr

# Explainability
import shap

# Visualization
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.manifold import TSNE, trustworthiness

# BioPython
from Bio import SeqIO

# Environment Settings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
np.random.seed(42)
warnings.filterwarnings('ignore')

# Filepaths for source data files
path1 = 'Downloads/uniprotkb_proteome_UP000002311_2026_07_02.fasta'
path2 = 'Downloads/BIOGRID-ORGANISM-LATEST.mitab/BIOGRID-ORGANISM-Saccharomyces_cerevisiae_S288c-5.0.259.mitab.txt'

In [ ]:
# New installations required for this project

#%pip install biopython
#pip install torch
#pip install safetensors==0.4.5
#pip install fair-esm

# Helper Functions

In this section we're defining some helper functions for use in the rest of the notebook.

In [ ]:
# Funtion to read the ID and Sequence info from the FASTA file
def read_fasta(file_path):
    sequences = {}
    current_id = None
    current_seq = []

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Lines starting with '>' indicate a new sequence ID
            if line.startswith(">"):
                if current_id:
                    sequences[current_id] = "".join(current_seq)
                current_id = line[1:]  # Remove the '>' character
                current_seq = []
            else:
                current_seq.append(line)
        
        # Add the final sequence from the file
        if current_id:
            sequences[current_id] = "".join(current_seq)
            
    return sequences


# Function to extract the Uniprot gene IDs from the Biogrid dataframe
def extract_locuslink(s):
    match = re.search(r'locuslink:([^|]+)', s)
    return match.group(1) if match else None


# Function to extract interactions from Biogrid dataframe
def get_interactors(bg, protein_id):
    # Protein appears in column A
    a_partners = bg.loc[
        bg['Gene_A'] == protein_id,
        'Gene_B'
    ]

    # Protein appears in column B
    b_partners = bg.loc[
        bg['Gene_B'] == protein_id,
        'Gene_A'
    ]

    # Combine, deduplicate, and return as a list
    return pd.concat([a_partners, b_partners]).drop_duplicates().tolist()


# Function to create the embedding for a given protein
def get_esm_embedding(model, sequence, batch_converter):
    batch_labels, batch_strs, batch_tokens = batch_converter([("P", sequence)])
    batch_tokens = batch_tokens.to(device)
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[last_layer], return_contacts=False)
    token_representations = results["representations"][last_layer]
    protein_representations = token_representations[0,1:-1,:].mean(dim=0).cpu().numpy()
    return protein_representations


# Function to generate a randome amino acid sequence of fixed length
def random_protein(length):
    rand01 = ''.join(random.choices("ACDEFGHIKLMNPQRSTVWY", k=length))
    return rand01


# Function to create the pair-wise feature set for 2 embeddings
def get_pairwise_features(emb1, emb2):
    pairwise = np.concatenate([embA, embB, np.abs(embA - embB), embA * embB])
    return pairwise

# Section 1 - Data Import

In this section we are importing amino acid sequences for all 6,067 amino acids in S. cerevisiae.

In [ ]:
# Import dataset 
for record in SeqIO.parse(path1, "fasta"):
    print(f"ID: {record.id}")
    print(f"Sequence: {record.seq}")
    print(f"Length: {len(record.seq)}\n")

In [ ]:
# Create the protein-sequence dictionary and a dictionary of protein residue length
sequence_dictionary = {}
length_dictionary = {}
fasta_data = read_fasta(path1)

for seq_id, sequence, in fasta_data.items():
    print(f"ID: {seq_id}\n" )# Sequence: {sequence}\n")
    
    match = re.search(r'GN=(\S+)', seq_id)

    if match:
        gene_name = match.group(1)
        print(gene_name)
        
        sequence_dictionary[gene_name] = sequence
        length_dictionary[gene_name] = len(sequence)

# Create a list of all proteins
gene_list = list(sequence_dictionary.keys())

In [ ]:
# Inspect the results
#print(gene_list)
len(gene_list)

In [ ]:
# Check if a particular gene is included in the Uniprot data
if 'CDC73' in gene_list:
    print("Item found!")

Here we are importing the interaction data from BioGrid.

In [ ]:
# Now extract the Biogrid data and create the Biogrid dataframe
bg = pd.read_csv(path2,
    sep="\t",
    header=None,
    dtype=str
)

print(bg.shape)

# Reset the row and column indices
bg.columns = bg.iloc[0]
bg = bg[1:]
bg.columns.name = None
bg = bg.reset_index(drop=True)

print(bg.shape)

In [ ]:
bg.head()

In [ ]:
# Check the Biogrid data at a particular location
print(bg.at[1, 'Alt IDs Interactor A'])

In [ ]:
# Extract the interactor A and B genes and create a new column for each
bg['Gene_A'] = bg['Alt IDs Interactor A'].apply(extract_locuslink)
bg['Gene_B'] = bg['Alt IDs Interactor B'].apply(extract_locuslink)

bg.head()

This cell takes quite a while to run locally; therefore writing the results to file for future use.

In [ ]:
# Now we make the interaction dictionary and write to file for future use
ia_dic = {}

# Iterate through each gene and extract interactions
for gene in gene_list:
    interactors = get_interactors(bg, gene)
    ia_dic[gene] = interactors

# Write the interaction dictionary to file 
with open('interactions.pkl', 'wb') as file:
    pickle.dump(ia_dic, file)

In [ ]:
# Read the interaction dictionary from the binary file
with open('interactions.pkl', 'rb') as file:
    interaction_dictionary = pickle.load(file)

In [ ]:
# Inspect the results
interaction_dictionary

# Section 2 - Data Cleaning & Pre-processing

In this section we perform some data cleaning and pre-processing operations. First, we need to make sure we have sequence and interaction data for all proteins.

In [ ]:
# Find entries in the interaction dictionary that are empty
empty_keys = [key for key, value in interaction_dictionary.items() if not value]
print('Number of empty entries in the interaction dictionary:')
print(len(empty_keys))

# Remove them from the sequence and interation dictionaries
for key in empty_keys:
    sequence_dictionary.pop(key, None)
    length_dictionary.pop(key, None)
    interaction_dictionary.pop(key, None)

print('')
print('Number of proteins for which he have sequence AND interaction data:')
print(len(sequence_dictionary))

new_gene_list = list(sequence_dictionary.keys())

In [ ]:
# Next we need to remove any interactors for which we don't have sequence data
interaction_dictionary_filtered = {
    protein: [p for p in partners if p in new_gene_list]
    for protein, partners in interaction_dictionary.items()
}

In [ ]:
interaction_dictionary_filtered

In [ ]:
# Finally we create the overall interaction matrix using the interaction dictionary - a square
# matrix 5670 x 5670 with 1's indicating interactions, 0's indicating no interaction

# We will call this the 'Master Interaction Matrix' or MIM
MIM = pd.DataFrame(0, index=new_gene_list, columns=new_gene_list)

for protein, partners in interaction_dictionary_filtered.items():
    MIM.loc[protein, partners] = 1

In [ ]:
# Perform some checks to make sure MIM is setup correctly, check sparsity
row = 'CET1'
column = 'STE2'

print(f'MIM entry at {row}, {column}:')
print(MIM.loc[row, column])
print('')

# See how sparse MIM 
count_ones = np.count_nonzero(MIM)
print('Number of non-zero entries in the MIM:')
print(count_ones)
print('')

print('Total number of entries in the MIM:')
num_entries = 5670*5670
print(num_entries)
print('')

print('Sparsity of the MIM:')
print(count_ones/num_entries)

# Section 3 - Create Embeddings

In this section we are creating the embeddings for all the proteins for whcih we have sequence AND interaction data. We will then perform some checks to make sure the embeddings are behaving as expected, by comparing the cosine similarity between various embeddings.

In [ ]:
# Load a pre-trained ESM embedding model for use in the create emmbedding function

# Utilize GPU resources if available, otherwise default to CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device {device} for data processing.')

# Load the model producing 1280-dim embeddings
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
last_layer = len(model.layers)
batch_converter = alphabet.get_batch_converter()
model.eval(); 
model.to(device)

# Print the token alphabet
alphabet.to_dict()

In [ ]:
# Perform some basic checks of the embedding

# Define a sequence and confirm the embedding has the right dimensionality
seq1 = 'MSDAAPSLSNLFYDPTYNPGQSTINYTSIYGNGSTITFDELQGLVNSTVTQAIMFGVRCGAAALTLIVMWMTSRSRKTPIFIINQVSLFLIILHSALYFKYLLSNYSSVTYALTGFPQFISRGDVHVYGATNIIQVLLVASIETSLVFQIKVIFTGDNFKRIGLMLTSISFTLGIATVTMYFVSAVKGMIVTYNDVSATQDKYFNASTILLASSINFMSFVLVVKLILAIRSRRFLGLKQFDSFHILLIMSCQSLLVPSIIFILAYSLKPNQGTDVLTTVATLLAVLSLPLSSMWATAANNASKTNTITSDFTTSTDRFYPGTLSSFQTDSINNDAKSSLRSRLYDLYPRRKETTSDKHSERTFVSETADDIEKNQFYQLPTPTSSKNTRIGPFADASYKEGEVEPVDMYTPDTAADEEARKFWTEDNNNL'
emb1 = get_esm_embedding(model, seq1, batch_converter)

# Confirm shape, content
print('Length of sequence embedding:')
print(len(emb1))
print('')
print('Values in embedding:')
print(emb1)

In [ ]:
# Perform some more advanced checks

# Define two similar proteins - human alpha globin (HAG) and human beta globin (HBG)
HAG = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"
HBG = "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH"

# Pick a random S. cerivisiae protein
CDC73 = sequence_dictionary['CDC73']

# Create 2 random proteins of similar length
rand1 = random_protein(150)
rand2 = random_protein(150)

# Check the cosine similarity between the two
alpha = get_esm_embedding(model, HAG, batch_converter)
beta = get_esm_embedding(model, HBG, batch_converter)
delta = get_esm_embedding(model, CDC73, batch_converter)
gamma1 = get_esm_embedding(model, rand1, batch_converter)
gamma2 = get_esm_embedding(model, rand2, batch_converter)

In [ ]:
# Calculate the cosine similarity betwen HAG and HBG
print("Cosine similarity between human alpha globin (HAG) and human beta globin (HBG) embeddings:")
a = alpha
b = beta
similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(similarity)
print('')

# Calculate the cosine similarity between HAG and a protein sequence from S. cerivisiae
print("Cosine similarity between human alpha globin (HAG) and CDC73:")
a = alpha
b = delta
similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(similarity)
print('')

# Calculate the cosine similarity between HAG and a random protein sequence
print("Cosine similarity between human alpha globin (HAG) and a random protein sequence:")
a = alpha
b = gamma1
similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(similarity)
print('')

# Calculate the cosine similarity between HAG and the other random protein sequence
print("Cosine similarity between human alpha globin (HAG) and another random protein sequence:")
a = alpha
b = gamma2
similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(similarity)

This cell takes quite a while to run locally; therefore writing the results to file for future use.

In [ ]:
# Creat a dictionary of embeddings for all proteins, and write to file
embedding_dict = {}

# Loop through each protein in the set
for gene in new_gene_list:
    sequence = sequence_dictionary[gene]
    embedding = get_esm_embedding(model, sequence, batch_converter)
    embedding_dict[gene] = embedding
    
# Write the embeddings dictionary to file 
with open('embeddings.pkl', 'wb') as file:
    pickle.dump(embedding_dict, file)

In [ ]:
# Read the embedding dictionary from the binary file
with open('embeddings.pkl', 'rb') as file:
    embedding_dict = pickle.load(file)

In [ ]:
embedding_dict

# Section 3 - Train the Model

In this section we begin training our NN model

In [ ]:
# First we need to create a test/train split
train_proteins, test_proteins = train_test_split(
    new_gene_list,
    test_size=0.20,   #Train on 80% of the data, test on the other 20%
    random_state=42
)

# Now form test and train interaction matrices
MIM_train = MIM.loc[train_proteins, train_proteins]
MIM_test = MIM.loc[test_proteins, test_proteins]

In [ ]:
def create_balanced_dataset(interaction_matrix, embeddings, n=1000, random_state=42):
    
    rng = np.random.default_rng(random_state)
    
    proteins = interaction_matrix.index.to_numpy()
    
    # Get upper-triangle indices, excluding diagonal
    i, j = np.triu_indices(len(proteins), k=1)
    
    # Get corresponding interaction values
    values = interaction_matrix.values[i, j]
    
    # Separate positive and negative pairs
    positive_idx = np.where(values == 1)[0]
    negative_idx = np.where(values == 0)[0]
    
    # Check that enough pairs exist
    if len(positive_idx) < n:
        raise ValueError(
            f"Only {len(positive_idx)} positive pairs available; "
            f"cannot sample {n}."
        )
    
    if len(negative_idx) < n:
        raise ValueError(
            f"Only {len(negative_idx)} negative pairs available; "
            f"cannot sample {n}."
        )
    
    # Randomly select n of each
    positive_sample = rng.choice(
        positive_idx, size=n, replace=False
    )
    
    negative_sample = rng.choice(
        negative_idx, size=n, replace=False
    )
    
    selected = np.concatenate([
        positive_sample,
        negative_sample
    ])
    
    # Shuffle positive and negative examples together
    rng.shuffle(selected)
    
    X = []
    y = []
    pairs = []
    
    for idx in selected:
        
        protein_a = proteins[i[idx]]
        protein_b = proteins[j[idx]]
        
        emb_a = embeddings[protein_a]
        emb_b = embeddings[protein_b]
        
        # Symmetric pair representation
        pair_embedding = np.concatenate([
            np.abs(emb_a - emb_b),
            emb_a * emb_b
        ]) # Could use alternate function "get_pairwise_features(emb1, emb2)" here
        
        X.append(pair_embedding)
        y.append(values[idx])
        pairs.append((protein_a, protein_b))
    
    return np.array(X), np.array(y), pairs

In [ ]:
# Create test and train dataframes
X_train, y_train, train_pairs = create_balanced_dataset(
    MIM_train,
    embedding_dict,
    n=100000,       # Number of positive and negative interactions to pull from the training interaction matrix
    random_state=42
)

X_test, y_test, train_pairs = create_balanced_dataset(
    MIM_test,
    embedding_dict,
    n=10000,        # Number of positive and negative interactions to pull from the test interaction matrix
    random_state=42
)

# Scale the train and test sets
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Define a simple neural network to start with
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.3),
    
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    
    layers.Dense(64, activation="relu"),
    
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.AUC(name="auc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

model.summary()

In [ ]:
# Train the simple neural network 
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Save it for future use
model.save("ppi_esm2_model.keras")

In [ ]:
# Re-load the previously trained model (as opposed to re-training)
model = keras.models.load_model("ppi_esm2_model.keras")

In [ ]:
# Evaluate the model
results = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

for name, value in zip(model.metrics_names, results):
    print(f"{name}: {value:.4f}")
    
results = model.evaluate(X_test, y_test, verbose=0, return_dict=True)

print(results)

In [ ]:
y_prob = model.predict(X_test).ravel()

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC :", average_precision_score(y_test, y_prob))

In [ ]:
# First we need to create the train / test split

# Use 10 different random seeds to create 10 unique test/train splits

rand_seed = [1]#, 1, 2, 3, 4, 7, 8, 9, 69, 420]
overall_score = []

for seed in rand_seed:    
    print()
    print()
    print('RANDOM SEED: ', seed)
    
    # CREATE THE TEST / TRAIN SPLIT
    # Find all 'races' consisting of pairs of event id and category name and save as a list
    unique_pairs = list(df2.groupby(['event.id','clean_categories.name']).groups.keys())

    # Split into test and train races
    train_pairs, test_pairs = train_test_split(
        unique_pairs,
        test_size=0.2,
        random_state=seed,   # for reproducibility
    )

    # Create test and train dataframes
    train_mask = np.logical_or.reduce([
        (df2['event.id'] == a) & (df2['clean_categories.name'] == b)
        for (a, b) in train_pairs
    ])
    df_train = df2[train_mask]

    test_mask = np.logical_or.reduce([
        (df2['event.id'] == a) & (df2['clean_categories.name'] == b)
        for (a, b) in test_pairs
    ])
    df_test = df2[test_mask]
    
    
    # FEATURE SELECTION
    # Create the actual X and y dataframes for training by selecting features of interest
    series_columns = [col for col in df2.columns if col.startswith('series_copy_')]
    feature_list = ['age',
                    'bib',
                    'sex_binary',
                    'race_dist_m',
                    'run_or_walk',
                    'counts.participants.registered'
                ] #+ series_columns

    # Create training data
    X_train = df_train[feature_list]
    y_train = df_train['total_seconds']
    
    
    # TRAIN THE MODEL
    # Train model #5 (Simple NN)

    # Scale the data
    scaler = StandardScaler()
    X_train5 = scaler.fit_transform(X_train)

    model5A = models.Sequential([
        layers.Input(shape=(X_train5.shape[1],)),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='linear')
    ])

    model5A.compile(
        optimizer='adam',
        loss='mean_absolute_error',
        metrics=['mean_absolute_error']
    )

    history = model5A.fit(
        X_train5, y_train,
        epochs=20,
        batch_size=16,
        verbose=1
    )
    
    # EVALUATE THE MODEL
    # Evaluate each group in the test dataset
    unique_pairs = df_test.groupby(['event.id','clean_categories.name']).groups.keys()

    # Initialize some stuff for scoring
    total_score = 0
    idx = 0

    # Loop through each event
    for pair in unique_pairs:
        event_id, dist = pair

        print(pair)
        # Create the dataframe for testing
        X_test = df_test[(df_test['event.id'] == event_id) & (df_test['clean_categories.name'] == dist)]
        X_test = X_test.dropna(subset=['overall_ranking'])
        X_test1 = X_test[feature_list]


        # Establish y_test (2 different ways)
        #y_test = X_test['overall_ranking'].to_frame()
        y_test = pd.DataFrame(X_test['overall_ranking'].rank(),X_test.index)


        # Handle NaN in the feature set
        X_test2 = X_test1.fillna(value=-1)

        # Scale the data
        X_test3 = scaler.transform(X_test2)


        # Make the predictions and add to X_test2 dataframe
        predictions = model5A.predict(X_test3).flatten()

        # Add predicted finishing times to the unscaled dataframe
        X_test2['predicted_time'] = predictions
        # Convert time to finishing position
        X_test2['predicted_ranking'] = (
        X_test2['predicted_time']
          .rank(method='first', ascending=True)
          .astype(int)
        )

        # Extract y_pred for scoring
        y_pred = X_test2['predicted_ranking'].to_frame()

        # Calculate and print score
        scr = score(y_test, y_pred)
        total_score = total_score + scr
        idx = idx + 1
        print(scr)

    print('Overall Average:', total_score/idx)
    overall_score.append(total_score/idx)

print()
print()

print(overall_score)
final_score = np.mean(overall_score)
print('Final Score:', final_score)

# Section 4 - Evaluate the Model

In this section we are